# Ekstraksi dan Menampilkan Matriks Kernel

Notebook ini dibuat untuk memuat file `kernel_history.pkl` (atau `kernel_history_all.pkl`) dan menampilkan seluruh nilai matriks untuk setiap filter pada setiap channel di semua epoch.

In [6]:
import pickle
import numpy as np
import torch
import os

# Deteksi file history kernel
file_path = 'kernel_history.pkl'
if not os.path.exists(file_path):
    file_path = 'kernel_history_all.pkl'

print(f"Memuat data dari: {file_path} ...")
with open(file_path, 'rb') as f:
    kernel_history_all = pickle.load(f)

print("Berhasil memuat data!")
print(f"Jumlah layer yang terekam: {len(kernel_history_all)}")

Memuat data dari: kernel_history.pkl ...
Berhasil memuat data!
Jumlah layer yang terekam: 39


### Menampilkan Matriks (Satu Filter, Satu Channel) Tiap Blok

Kode di bawah ini akan menampilkan satu representasi spesifik untuk setiap blok: filter awal (0) dan channel awal (0) untuk semua epoch. Layer perwakilan yang diambil mencakup 11 blok mulai dari awal arsitektur hingga tahap klasifikasi:
`Layer -> Epoch -> Filter 0 -> Channel 0 -> Nilai Matriks`

In [7]:
import sys
import numpy as np
import torch
import pandas as pd
from IPython.display import display, HTML

np.set_printoptions(suppress=True)
torch.set_printoptions(profile="full", sci_mode=False)

def print_target_kernels_table(history_dict, target_filter_idx=0, target_channel_idx=0):
    """
    Menampilkan nilai matriks (kernel) dalam bentuk Tabel Pandas yang rapi.
    Hanya mencetak nilai matriks untuk layer perwakilan tiap blok,
    pada 1 filter awal dan 1 channel awal untuk semua epoch.
    """
    target_layers = [
        "model.0.conv",
        "model.1.conv",
        "model.2.m.0.cv1.conv",
        "model.3.conv",
        "model.4.m.0.cv1.conv",
        "model.5.conv",
        "model.6.m.0.cv1.conv",
        "model.7.conv",
        "model.8.m.0.cv1.conv",
        "model.9.m.0.attn.qkv.conv",
        "model.10.conv.conv"
    ]

    for layer_name in target_layers:
        if layer_name not in history_dict:
            print(f"Layer {layer_name} tidak ditemukan di history_dict.")
            continue

        epochs_data = history_dict[layer_name]

        # Header Blok
        display(HTML(f"<h3 style='background-color:#4CAF50; color:white; padding:10px;'>LAYER/BLOK: {layer_name.upper()}</h3>"))

        # Buat penampung semua epoch agar bisa di-display berjajar (Opsional) atau ke bawah
        for epoch_idx, epoch_tensor in enumerate(epochs_data):
            epoch_np = epoch_tensor.detach().cpu().numpy()

            ndim = epoch_np.ndim
            num_filters = epoch_np.shape[0]
            num_channels = epoch_np.shape[1] if ndim > 1 else 1

            if target_filter_idx >= num_filters:
                print(f"-> Layer ini hanya memiliki {num_filters} filter. Skipped.")
                continue

            # Ekstrak Matriks 2D
            if ndim >= 2 and target_channel_idx < num_channels:
                if ndim >= 3:
                    matrix = epoch_np[target_filter_idx, target_channel_idx]
                else:
                    matrix = epoch_np[target_filter_idx]
            else:
                matrix = epoch_np[target_filter_idx]
                if matrix.ndim == 1:
                    matrix = np.expand_dims(matrix, axis=0) # Paksa 2D

            # Konversi matriks numpy menjadi Pandas DataFrame untuk tampilan Rapi
            df = pd.DataFrame(matrix)

            # Print informasi Epoch di atas tabel
            display(HTML(f"<b>--- EPOCH {epoch_idx + 1} (Filter {target_filter_idx}, Channel {target_channel_idx}) ---</b>"))

            # Tampilkan dataframe dengan styling warna heatmap / gradasi
            # Menggunakan cmap coolwarm atau Blues
            styled_df = df.style.format("{:.4f}").background_gradient(cmap='coolwarm', axis=None)
            display(styled_df)

        display(HTML("<hr>")) # Garis pemisah antar layer

print(f"Mencetak nilai matriks untuk Filter 0 dan Channel 0 pada setiap epoch dalam bentuk Tabel...")
print_target_kernels_table(kernel_history_all, target_filter_idx=0, target_channel_idx=0)


Mencetak nilai matriks untuk Filter 0 dan Channel 0 pada setiap epoch dalam bentuk Tabel...


,0,1,2
0,-0.0634,0.0829,-0.0671
1,0.1856,0.0560,-0.1685
2,0.0263,-0.3206,0.3336


,0,1,2
0,-0.0627,0.0834,-0.0664
1,0.1860,0.0565,-0.1677
2,0.0267,-0.3200,0.3344


,0,1,2
0,-0.0623,0.0838,-0.0660
1,0.1860,0.0567,-0.1674
2,0.0267,-0.3198,0.3348


,0,1,2
0,-0.0622,0.0839,-0.0657
1,0.1861,0.0567,-0.1673
2,0.0270,-0.3194,0.3351


...

,0
0,0.0876


,0
0,0.0881


,0
0,0.0887


,0
0,0.0891


,0
0,0.0895


### Opsi Tambahan: Menyimpan ke File Teks
Jika output di Jupyter Notebook lag/crash karena terlalu panjang, Anda bisa menjalankan sel di bawah ini untuk mengekspor (save) seluruh nilai tersebut ke file `TXT` per layer.

In [ ]:
import os
import numpy as np
from multiprocessing import Pool
from io import StringIO


def _write_layer_worker(args):
    layer_name, epochs_list, export_dir = args

    safe_name = layer_name.replace(".", "_")
    file_path = os.path.join(export_dir, f"{safe_name}.txt")

    buf = StringIO()
    buf.write("=" * 50 + "\n")
    buf.write(f"LAYER: {layer_name}\n")
    buf.write("=" * 50 + "\n")

    target_filter_idx = 0
    target_channel_idx = 0

    for epoch_idx, epoch_np in enumerate(epochs_list):
        buf.write(f"\n--- EPOCH {epoch_idx + 1} ---\n")
        ndim = epoch_np.ndim
        num_filters = epoch_np.shape[0]
        num_channels = epoch_np.shape[1] if ndim > 1 else 1

        if target_filter_idx < num_filters:
            buf.write(f"\n  [Filter {target_filter_idx}]\n")
            if target_channel_idx < num_channels:
                buf.write(f"    > Channel {target_channel_idx}:\n")
                if ndim >= 3:
                    matrix = epoch_np[target_filter_idx, target_channel_idx]
                else:
                    matrix = epoch_np[target_filter_idx]
            else:
                buf.write(f"    > Channel 0:\n")
                matrix = epoch_np[target_filter_idx]

            matrix_str = np.array2string(matrix, separator=", ", prefix=" " * 6)
            buf.write(f"      {matrix_str}\n")

    # Single write
    with open(file_path, "w", encoding="utf-8", buffering=1024 * 1024) as f:
        f.write(buf.getvalue())

    return layer_name


def export_kernels_to_txt_fast(history_dict, export_dir="kernels"):
    os.makedirs(export_dir, exist_ok=True)

    target_layers = [
        "model.0.conv",
        "model.1.conv",
        "model.2.m.0.cv1.conv",
        "model.3.conv",
        "model.4.m.0.cv1.conv",
        "model.5.conv",
        "model.6.m.0.cv1.conv",
        "model.7.conv",
        "model.8.m.0.cv1.conv",
        "model.9.m.0.attn.qkv.conv",
        "model.10.conv.conv"
    ]

    tasks = []
    for layer_name in target_layers:
        if layer_name in history_dict:
            epochs_data = history_dict[layer_name]
            epochs_np = [t.detach().cpu().numpy() for t in epochs_data]
            tasks.append((layer_name, epochs_np, export_dir))

    print(f"Mengekspor {len(tasks)} layer (1 Filter, 1 Channel) dengan proses paralel...")

    with Pool(processes=min(len(tasks), os.cpu_count() or 1)) as pool:
        for layer_name in pool.imap_unordered(_write_layer_worker, tasks):
            print(f"  ✓ {layer_name}")

    print(f"\nSelesai! Export ke: {export_dir}")


export_kernels_to_txt_fast(kernel_history_all)


Mengekspor 39 layer dengan 16 proses...
  ✓ model.0.conv
  ✓ model.2.m.0.cv1.conv
  ✓ model.2.m.0.cv2.conv
  ✓ model.1.conv
  ✓ model.4.m.0.cv2.conv
  ✓ model.2.cv1.conv
  ✓ model.4.m.0.cv1.conv
  ✓ model.6.m.0.m.0.cv2.conv
  ✓ model.6.m.0.cv1.conv
  ✓ model.6.m.0.cv2.conv
  ✓ model.6.m.0.m.0.cv1.conv
  ✓ model.6.m.0.m.1.cv2.conv
  ✓ model.2.cv2.conv
  ✓ model.6.m.0.m.1.cv1.conv
  ✓ model.4.cv1.conv
  ✓ model.6.m.0.cv3.conv
  ✓ model.3.conv
  ✓ model.8.m.0.cv2.conv
  ✓ model.8.m.0.cv1.conv
  ✓ model.8.m.0.m.0.cv2.conv
  ✓ model.9.m.0.attn.pe.conv
  ✓ model.8.m.0.m.0.cv1.conv
  ✓ model.8.m.0.m.1.cv1.conv
  ✓ model.4.cv2.conv
  ✓ model.8.m.0.m.1.cv2.conv
  ✓ model.6.cv1.conv
  ✓ model.8.m.0.cv3.conv
  ✓ model.6.cv2.conv
  ✓ model.9.m.0.attn.proj.conv
  ✓ model.5.conv
  ✓ model.9.m.0.ffn.0.conv
  ✓ model.9.m.0.attn.qkv.conv
  ✓ model.9.m.0.ffn.1.conv
  ✓ model.7.conv
  ✓ model.9.cv1.conv
  ✓ model.8.cv1.conv
  ✓ model.9.cv2.conv
  ✓ model.8.cv2.conv
  ✓ model.10.conv.conv

Selesai! Export